In [10]:
#%pip install paddleocr
#%pip install paddlepaddle==3.2.2
#%pip install tqdm
#%pip install ultralytics
#%pip install pandas
#%pip install scikit-learn

from paddleocr import PaddleOCR as PaddleOCRModel
from ultralytics import YOLO
import pandas as pd
import re


In [13]:
from pathlib import Path
import os

# Run once
os.chdir(Path.cwd().parent)


## Extract crops

In [ ]:
class yolo:
    def __init__(self, model_path):
        self.model = YOLO(model_path)

    def predict_and_save(self, img_path, save_path):
        # Input folder path, run inference over all images in the folder, and save crops
        self.model.predict(img_path,
                        imgsz=896,
                        device=0,
                        conf=0.6,
                        save_crop = True,
                        project = save_path
                        ) 
        
        

In [16]:
yolo_model = yolo("models/customYOLO26N.pt")

yolo_model.predict_save("data/rawImages/test", "/home/daniel/Documents/deepLearning/Project2_TrOCR/data/OCRcrops/test")

yolo_model.predict_save("data/rawImages/train", "/home/daniel/Documents/deepLearning/Project2_TrOCR/data/OCRcrops/train")

yolo_model.predict_save("data/rawImages/val", "/home/daniel/Documents/deepLearning/Project2_TrOCR/data/OCRcrops/val")


image 1/9 /home/daniel/Documents/deepLearning/Project2_TrOCR/data/rawImages/test/IMG_0846_jpeg.rf.9a793b4384afca75ab9a22146e6fcebd.jpg: 896x672 10 rotors, 17.3ms
image 2/9 /home/daniel/Documents/deepLearning/Project2_TrOCR/data/rawImages/test/IMG_0848_jpeg.rf.d6b99bc6ba421916811371f16dece3fa.jpg: 896x672 39 rotors, 18.0ms
image 3/9 /home/daniel/Documents/deepLearning/Project2_TrOCR/data/rawImages/test/IMG_0849_jpeg.rf.fa71aa8415322bf4f7c5fcbaf9302d58.jpg: 896x672 24 rotors, 18.2ms
image 4/9 /home/daniel/Documents/deepLearning/Project2_TrOCR/data/rawImages/test/IMG_0852_jpeg.rf.f8b79984537a484a1539b67b61ca18a2.jpg: 896x672 19 rotors, 15.4ms
image 5/9 /home/daniel/Documents/deepLearning/Project2_TrOCR/data/rawImages/test/IMG_0853_jpeg.rf.6826179f74082027ed6959aa9273960b.jpg: 896x672 11 rotors, 18.3ms
image 6/9 /home/daniel/Documents/deepLearning/Project2_TrOCR/data/rawImages/test/IMG_0855_jpeg.rf.80474a7b18d63c1f595eeede6e4a8f8d.jpg: 896x672 35 rotors, 18.1ms
image 7/9 /home/daniel/Docu

## Extract baseline outputs

In [18]:
class OCRpipeline:
    def __init__(self, text_detection_model_name = "PP-OCRv5_mobile_det", text_recognition_model_name = "PP-OCRv5_mobile_rec", use_doc_orientation_classify = False, use_doc_unwarping = False, use_textline_orientation = False):
        self.ocr_model = PaddleOCRModel(text_detection_model_name=text_detection_model_name,
                                    text_recognition_model_name=text_recognition_model_name,
                                    use_doc_orientation_classify=use_doc_orientation_classify,
                                    use_doc_unwarping=use_doc_unwarping,
                                    use_textline_orientation=use_textline_orientation)
        
    def paddleInference(self, img_path):
        # Input image path, run OCR and return text
        result = self.ocr_model.predict(img_path)
        return result
    

    def order_best_text(self, results, confidence_threshold = 0.5):
        rec_texts = results[0]["rec_texts"]
        rec_scores = results[0]["rec_scores"]

        # Pair text with score
        text_score_pairs = list(zip(rec_texts, rec_scores))

        # Filter out pairs with low confidence (e.g., score < 0.5)
        filtered_pairs = [pair for pair in text_score_pairs if pair[1] >= confidence_threshold]

        # Sort by score (highest first)
        text_score_pairs_sorted = sorted(filtered_pairs, key=lambda x: x[1], reverse=True)

        return text_score_pairs_sorted if text_score_pairs_sorted else None
    

    def extract_best_text(self, text_score_pairs_sorted, pattern = re.compile(r'^\d{4,5}(DL)?$')):
        # Extract the best pattern matching text from the sorted pairs
        pattern = pattern

        if text_score_pairs_sorted is None:
            return "null"
        
        for text, score in text_score_pairs_sorted:
            text = text.strip().upper()
        
            # Must match part-number pattern
            if pattern.match(text):
                return text
            
        return "null"


            




In [ ]:
import os
import csv
import re
from pathlib import Path

folder_paths = ["data/OCRcrops/test/rawCropTest"]

output_files = ["ocrResultsTest.csv"]

# Function to extract numeric keys for natural ordering
def natural_key(path):
    match = re.match(r"IMG_(\d+)_jpeg(\d*)\.\w+", path.name, re.IGNORECASE)
    if match:
        img_num = int(match.group(1))
        jpeg_str = match.group(2)
        jpeg_num = int(jpeg_str) if jpeg_str else 1
        return (img_num, jpeg_num)
    else:
        return (float('inf'), float('inf'))

# Loop over folders and output files
for folder_path, output_file in zip(folder_paths, output_files):
    self = OCRpipeline()
    folder = Path(folder_path)

    # Get all image files and sort naturally
    image_files = sorted(
        [f for f in folder.glob("*") if f.suffix.lower() in [".png", ".jpg", ".jpeg", ".bmp", ".webp"]],
        key=natural_key
    )

    # Write results to CSV
    with open(output_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["filename", "best_text"])

        for img_path in image_files:
            ocr_results = self.paddleInference(str(img_path))
            text_score_pairs_sorted = self.order_best_text(ocr_results)
            best_text = self.extract_best_text(text_score_pairs_sorted)
            writer.writerow([img_path.name, best_text])

Creating model: ('PP-OCRv5_mobile_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/daniel/.paddlex/official_models/PP-OCRv5_mobile_det`.
Creating model: ('PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/daniel/.paddlex/official_models/PP-OCRv5_mobile_rec`.


In [29]:
import os
import csv
import re
from pathlib import Path

folder_paths = ["data/OCRcrops/train/rawCropTrain",
                "data/OCRcrops/val/rawCropVal"]

output_files = ["ocrResultsTrain.csv",
                "ocrResultsVal.csv"]

# Function to extract numeric keys for natural ordering
def natural_key(path):
    match = re.match(r"IMG_(\d+)_jpeg_jpeg(\d*)\.\w+", path.name, re.IGNORECASE)
    if match:
        img_num = int(match.group(1))
        jpeg_str = match.group(2)
        jpeg_num = int(jpeg_str) if jpeg_str else 1
        return (img_num, jpeg_num)
    else:
        return (float('inf'), float('inf'))

# Loop over folders and output files
for folder_path, output_file in zip(folder_paths, output_files):
    self = OCRpipeline()
    folder = Path(folder_path)

    # Get all image files and sort naturally
    image_files = sorted(
        [f for f in folder.glob("*") if f.suffix.lower() in [".png", ".jpg", ".jpeg", ".bmp", ".webp"]],
        key=natural_key
    )

    # Write results to CSV
    with open(output_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["filename", "best_text"])

        for img_path in image_files:
            ocr_results = self.paddleInference(str(img_path))
            text_score_pairs_sorted = self.order_best_text(ocr_results)
            best_text = self.extract_best_text(text_score_pairs_sorted)
            writer.writerow([img_path.name, best_text])

Creating model: ('PP-OCRv5_mobile_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/daniel/.paddlex/official_models/PP-OCRv5_mobile_det`.


Creating model: ('PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/daniel/.paddlex/official_models/PP-OCRv5_mobile_rec`.
Creating model: ('PP-OCRv5_mobile_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/daniel/.paddlex/official_models/PP-OCRv5_mobile_det`.
Creating model: ('PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/daniel/.paddlex/official_models/PP-OCRv5_mobile_rec`.
